# M12 — locked CIFAR-100 test confirmation on Kaggle

This is the Kaggle-native runner for the final controlled RanPAC/random-ReLU confirmation. It preserves the same M12 authorization boundary and never selects a method from test accuracy.

Before running:

1. In **Settings**, set **Accelerator = GPU**. A single P100 is preferred; the code deliberately uses one GPU.
2. Set **Internet = On** for the pinned Git checkout, dependency installation, and checkpoint fallback.
3. In **Add Input**, attach `zaphat206/cifar-100`.
4. Kaggle automatically extracts recognized archives. To preserve the three source ZIPs byte-for-byte, rename each locally by appending `.bin` (for example, `artifact.zip` becomes `artifact.zip.bin`) without extracting or recompressing it. Upload the three `.zip.bin` files to one private Kaggle Dataset and attach it here.
5. Optionally attach the verified `model.safetensors`; otherwise it is downloaded from Hugging Face.
6. Use **Save Version → Save & Run All** for the long run. Kaggle executes that version top-to-bottom in a clean session and retains files written under `/kaggle/working`.

In [ ]:
# Immutable identities and Kaggle paths. Do not edit experimental choices.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='ec82d23bfd916e31475ef90459f78931895b8a22'
WORK_DIR='/kaggle/temp/SOHO-CL'
FEATURE_CACHE_DIR='/kaggle/temp/srq_m12_cifar_features'
OUTPUT_DIR='/kaggle/working/srq_m12_locked_output'
AUTHORIZATION='/kaggle/working/srq_m12_authorization.json'
EXPORT_PATH='/kaggle/working/srq_generalization_m12_locked_test_confirmation.zip'
CONFIG='configs/srq_generalization_m12_locked_test_confirmation.json'
RUNNER='tools/srq_generalization_m12.py'
SOURCE_NAMES={
 'm6':'srq_generalization_m6_width_sweep_train_only.zip',
 'm11':'srq_generalization_m11_adaptive_precision_train_only.zip',
 'm11b':'srq_generalization_m11b_scale_refined_train_only.zip'}
SOURCE_SHA={
 'm6':'b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e',
 'm11':'65ce03df4da7041833014628b59aac1167f77bde2d64348b9a8a1e4fe09370a7',
 'm11b':'f33153c24716a7c660044cacc1e56cf040ee63d314fd13be2c7d47cf4895a9cf'}
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Pinned source checkout, dependencies, GPU, and source hashes.
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
assert Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir(),'This notebook must run on Kaggle.'
repo=Path(WORK_DIR);repo.parent.mkdir(parents=True,exist_ok=True)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--no-checkout',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Settings -> Accelerator -> GPU, then restart the session.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m12_locked_test_confirmation.json':'644f1e68dfcd235bef986944b72b94ef0a3433faa74d7a68149e85c2022fde40',
 'tools/srq_generalization_m12.py':'25c4d35bae98c7a324a4dc73e13a6eb91a6eb8e05290c58627093c76a9b98583',
 'methods/analytic_ridge/adaptive_upper.py':'d34eae6072ea3e8736b6fb940c7a0691c845d8217569c7b6fe33f4b9e143c5dc',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'methods/analytic_ridge/__init__.py':'59babe9c4881a0991a7f5825958cd0ef8c41f4d6b88b1b7cfc843b6e9ba99aed',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
actual_commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
assert actual_commit==REPO_COMMIT,(actual_commit,REPO_COMMIT)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Pinned repository is dirty.'
gpu_names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('GPU(S):',gpu_names,'| runner uses cuda:0 only')
print('PINNED COMMIT/SOURCE LOCK: PASS',actual_commit)

In [ ]:
# Discover byte-preserved source artifacts, verify them, then restore .zip names in /kaggle/temp.
INPUT_ROOT=Path('/kaggle/input')
SOURCE_STAGE=Path('/kaggle/temp/srq_m12_source_artifacts')
SOURCE_STAGE.mkdir(parents=True,exist_ok=True)
def unique_preserved_archive(name):
 # Kaggle extracts .zip inputs server-side, so .zip.bin is the canonical upload form.
 accepted=(name,name+'.bin')
 matches=sorted(path for candidate in accepted for path in INPUT_ROOT.rglob(candidate) if path.is_file())
 assert len(matches)==1,(f'Expected exactly one byte-preserved {name}. Upload it as {name}.bin; found {matches}')
 return matches[0]
SOURCE_PATHS={}
for key,name in SOURCE_NAMES.items():
 uploaded=unique_preserved_archive(name)
 assert sha_raw(uploaded)==SOURCE_SHA[key],(uploaded.name,sha_raw(uploaded),SOURCE_SHA[key])
 staged=SOURCE_STAGE/name
 shutil.copyfile(uploaded,staged)
 assert sha_raw(staged)==SOURCE_SHA[key],(name,sha_raw(staged),SOURCE_SHA[key])
 SOURCE_PATHS[key]=str(staged)
cifar_dirs=sorted({path.parent for path in INPUT_ROOT.rglob('meta') if path.is_file() and (path.parent/'train').is_file() and (path.parent/'test').is_file()})
assert len(cifar_dirs)==1,f'Attach zaphat206/cifar-100 exactly once; raw CIFAR directories found: {cifar_dirs}'
CIFAR_ROOT=str(cifar_dirs[0])
checkpoint_candidates=[path for path in INPUT_ROOT.rglob('model.safetensors') if path.is_file() and path.stat().st_size==CHECKPOINT_SIZE and sha_raw(path)==CHECKPOINT_SHA]
if checkpoint_candidates:
 assert len(checkpoint_candidates)==1,f'Multiple matching checkpoints: {checkpoint_candidates}'
 CHECKPOINT_PATH=str(checkpoint_candidates[0])
else:
 from huggingface_hub import hf_hub_download
 CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE and sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
print('CIFAR ROOT:',CIFAR_ROOT)
print('CHECKPOINT:',CHECKPOINT_PATH)
print('THREE BYTE-PRESERVED SOURCE ARTIFACTS + DATA + CHECKPOINT: PASS')

In [ ]:
# Focused correctness tests before any CIFAR feature extraction.
subprocess.run([sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m12.py','tests/test_srq_generalization_m11.py','tests/test_srq_generalization_m6.py'],check=True)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Tests changed the pinned checkout.'
print('M12 KAGGLE PREFLIGHT TESTS: PASS')

In [ ]:
# Materialize TRAIN features only. Official test images remain unopened.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
 command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/kaggle/working/unused_m12','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
 subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
print('TRAIN-ONLY CACHE READY; test.pt ABSENT')

In [ ]:
# Freeze source/artifact/train identities before crossing the test boundary.
COMMON=['--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--source-m6-artifact',SOURCE_PATHS['m6'],'--source-m11-artifact',SOURCE_PATHS['m11'],'--source-m11b-artifact',SOURCE_PATHS['m11b'],'--authorization',AUTHORIZATION,'--require-clean-git']
subprocess.run([sys.executable,'-u',RUNNER,'authorize',*COMMON],check=True)
authorization=json.loads(Path(AUTHORIZATION).read_text())
print('LOCKED AUTHORIZATION:',authorization['authorization_id'])

## Authorized test boundary

Everything above is train-only. The next cell is the first operation permitted to materialize official CIFAR-100 test features. Do not change widths, methods, seeds, precision policy, Ridge, or retry decisions below this point.

In [ ]:
# Extract official TEST features only under the immutable authorization.
subprocess.run([sys.executable,'-u',RUNNER,'extract-test',*COMMON,'--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)],check=True)
assert (Path(FEATURE_CACHE_DIR)/'test.pt').is_file()
print('AUTHORIZED TEST CACHE READY')

In [ ]:
# Run 36 frozen units sequentially. Completed identity-matched unit JSON files resume safely.
print('M12 START: 6 paired replicates x 2 widths x 3 frozen methods.',flush=True)
completed=subprocess.run([sys.executable,'-u',RUNNER,'run',*COMMON,'--output-dir',OUTPUT_DIR,'--device','cuda'])
result_path=Path(OUTPUT_DIR)/'m12_results.json'
assert result_path.is_file(),'M12 stopped before writing its locked result; preserve the complete log and /kaggle/working files.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='COMPLETE_M12_LOCKED_TEST_CONFIRMATION','M12 integrity failed; do not retry based on accuracy.'

In [ ]:
# Descriptive locked table. Accuracy is reported but never gated.
import pandas as pd
rows=[]
for width_result in result['aggregate']:
 for method,metrics in width_result['methods'].items():
  rows.append({'width':width_result['width'],'method':method,'AIA mean':metrics['average_incremental_accuracy_percent']['mean'],'AIA SD':metrics['average_incremental_accuracy_percent']['sample_standard_deviation'],'Final mean':metrics['final_accuracy_percent']['mean'],'Final SD':metrics['final_accuracy_percent']['sample_standard_deviation'],'AIA minus Exact':metrics['paired_aia_difference_from_exact_pp']['mean'],'State MiB':metrics['final_total_persistent_bytes']/2**20,'Update seconds':metrics['analytic_update_seconds']['mean']})
display(pd.DataFrame(rows))

In [ ]:
# Paper-ready accuracy/state and task-trajectory figures.
import matplotlib.pyplot as plt
labels={'exact':'Exact','p2b_int8':'P2B INT8/FP32','adaptive_int8_fp16':'Adaptive INT8/FP16'}
colors={'exact':'#333333','p2b_int8':'#2878B5','adaptive_int8_fp16':'#D95319'}
fig,ax=plt.subplots(figsize=(6.2,4.2))
for wr in result['aggregate']:
 for method,metrics in wr['methods'].items():
  x=metrics['final_total_persistent_bytes']/2**20; y=metrics['average_incremental_accuracy_percent']['mean']
  ax.scatter(x,y,s=55,color=colors[method]);ax.annotate(f"{labels[method]} {wr['width']//1000}k",(x,y),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.set_xlabel('Persistent state (MiB)');ax.set_ylabel('Test AIA (%)');ax.grid(alpha=.25);fig.tight_layout();fig.savefig(Path(OUTPUT_DIR)/'m12_accuracy_state.svg')
fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for axis,width in zip(axes,[10000,20000]):
 for method in labels:
  selected=[u for u in result['units'] if u['identity']['width']==width and u['identity']['method']==method]
  means=[sum(u['records'][task]['test_accuracy_percent'] for u in selected)/len(selected) for task in range(10)]
  axis.plot(range(1,11),means,label=labels[method],color=colors[method])
 axis.set_title(f'Width {width:,}');axis.set_xlabel('Task');axis.grid(alpha=.25)
axes[0].set_ylabel('Seen-class test accuracy (%)');axes[1].legend(fontsize=8);fig.tight_layout();fig.savefig(Path(OUTPUT_DIR)/'m12_task_trajectory.svg')
plt.show()

In [ ]:
# Compact auditable artifact. Feature caches and immutable source ZIPs are excluded.
export=Path(EXPORT_PATH)
members=[Path(OUTPUT_DIR)/'m12_results.json',Path(OUTPUT_DIR)/'m12_summary.csv',Path(OUTPUT_DIR)/'m12_accuracy_state.svg',Path(OUTPUT_DIR)/'m12_task_trajectory.svg',Path(AUTHORIZATION),Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M12_PROTOCOL.md')]
members.extend(sorted((Path(OUTPUT_DIR)/'units').glob('*.json')))
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
 for path in members:
  arcname=('units/'+path.name) if path.parent.name=='units' else path.name
  archive.write(path,arcname);manifest[arcname]=sha_raw(path)
 archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M12 locked test confirmation','files':manifest},indent=2)+'\n')
assert export.is_file() and zipfile.is_zipfile(export)
print('FINAL ARTIFACT:',export)
print('SHA-256:',sha_raw(export),'| bytes:',export.stat().st_size)
from IPython.display import FileLink,display
display(FileLink(str(export)))